# rebound_state_head_lift_guard_v1

Purpose: audit whether low-precision rebound state-head detections actually create harmful target-position lift.

Context:
- `rebound_state_head_target_recipe_v1` passed a pre-PPO recipe gate using `momentum_drawdown` threshold `0.5`, but precision was only `0.054` with `123` false positives.
- This notebook adds effective false-positive lift metrics: a false positive that does not change the target position is not the same risk as a false positive that raises exposure.
- The result is still pre-PPO only; no runtime promotion can come from this screen.

External references:
- Qlib: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL train/validation/test discipline: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Supervised contrastive / representation decomposition: https://arxiv.org/abs/2004.11362
- Contrastive predictive coding representation learning: https://arxiv.org/abs/1807.03748
- Class imbalance caution: https://arxiv.org/abs/1901.05555

Governance:
- Baseline: `rebound_state_head_target_recipe_v1`.
- Role: diagnose / recipe guard.
- Changed layer: model-selection target routing guard, not runtime execution.
- Acceptance: allow future PPO recipe only if effective false-positive lift is bounded and all fixed-split / walk-forward / `+30pp` / `<=15% MaxDD` gates remain required.


In [1]:

from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'AGENTS.md').exists() and (candidate / 'src' / 'soxl_quant').exists():
            return candidate
    raise RuntimeError(f'Cannot locate repo root from {start}')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.soxl_quant.app.representation_diagnostics import (
    StateHeadTargetRecipeConfig,
    score_state_head_target_recipes,
)
from src.soxl_quant.policy.policy_builder import state_objective_target_position_from_row

EXPERIMENT = 'rebound_state_head_lift_guard_v1'
OUT_DIR = REPO_ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGETS_PATH = REPO_ROOT / 'runtime' / 'model_selection' / 'card' / 'state_objective_fold_native_teacher_router_v1' / 'fold_native_teacher_targets.csv'
RECIPE_SELECTION = REPO_ROOT / 'runtime' / 'model_selection' / 'card' / 'rebound_state_head_target_recipe_v1' / 'selection.json'
REPRESENTATION_SELECTION = REPO_ROOT / 'runtime' / 'model_selection' / 'card' / 'rebound_representation_separability_v1' / 'selection.json'

with RECIPE_SELECTION.open() as fh:
    recipe_selection = json.load(fh)
with REPRESENTATION_SELECTION.open() as fh:
    representation_selection = json.load(fh)

targets = pd.read_csv(TARGETS_PATH)
targets['datetime'] = pd.to_datetime(targets['datetime'])
targets['fold'] = targets['fold'].astype(str)
targets['is_rebound_cluster'] = targets['is_rebound_cluster'].fillna(False).astype(bool)
targets['is_downtrend_state'] = targets['is_downtrend_state'].fillna(False).astype(bool)
targets['true_target_position'] = [state_objective_target_position_from_row(row) for row in targets.to_dict('records')]
targets['default_target_position'] = np.where(
    targets['is_downtrend_state'],
    pd.to_numeric(targets['risk_target_position'], errors='coerce'),
    pd.to_numeric(targets['teacher_position'], errors='coerce'),
)

base_context = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'source_targets': str(TARGETS_PATH.relative_to(REPO_ROOT)),
    'rows': int(len(targets)),
    'rebound_rows': int(targets['is_rebound_cluster'].sum()),
    'baseline_recipe_precision': float(recipe_selection['best_state_precision']),
    'baseline_recipe_false_positive': int(recipe_selection['best_false_positive']),
    'baseline_recipe_fpr': float(recipe_selection['best_false_positive_rate']),
    'representation_feature_set': representation_selection['best_feature_set'],
}
print(json.dumps(base_context, indent=2, sort_keys=True))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{
  "baseline_recipe_false_positive": 123,
  "baseline_recipe_fpr": 0.09808612440191387,
  "baseline_recipe_precision": 0.05384615384615385,
  "created_at": "2026-05-25T13:53:31.630824+00:00",
  "experiment": "rebound_state_head_lift_guard_v1",
  "rebound_rows": 7,
  "representation_feature_set": "momentum_drawdown",
  "rows": 1261,
  "source_targets": "runtime/model_selection/card/state_objective_fold_native_teacher_router_v1/fold_native_teacher_targets.csv"
}


In [2]:

feature_sets = {
    'momentum_drawdown': ['mom_5', 'mom_21', 'drawdown_63'],
    'momentum_soxx_relative': ['mom_5', 'mom_21', 'drawdown_63', 'soxl_soxx_ratio_widen_21', 'soxx_mom_21'],
    'all_candidate_features': [
        'mom_5', 'mom_21', 'drawdown_63', 'rv_21', 'vol_of_vol_21',
        'soxl_soxx_ratio_widen_21', 'soxl_soxx_ratio_z_252', 'soxx_mom_21',
        'base_position', 'upside_position', 'router_position',
    ],
}
config = StateHeadTargetRecipeConfig(
    min_recall=0.70,
    min_precision=0.05,
    max_false_positive_rate=0.20,
    min_effective_lift=0.01,
    max_effective_false_positive_rate=0.01,
    max_false_positive_lift_mean=0.01,
    max_false_positive_lift_max=0.20,
    max_target_mae=0.02,
    max_rebound_mae=0.03,
    min_covered_folds=2,
)
result = score_state_head_target_recipes(
    targets,
    label_col='is_rebound_cluster',
    fold_col='fold',
    true_target_col='true_target_position',
    default_target_col='default_target_position',
    state_target_col='rebound_target_position',
    feature_sets=feature_sets,
    thresholds=[-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0],
    min_positive_train=2,
    min_positive_eval=2,
    config=config,
)
recipe_scores = result.recipe_scores
row_scores = result.row_scores
recipe_scores.to_csv(OUT_DIR / 'guarded_recipe_scores.csv', index=False)

best = recipe_scores.iloc[0]
best_rows = row_scores[(row_scores['feature_set'] == best['feature_set']) & (row_scores['threshold'] == best['threshold'])].copy()
best_rows['effective_false_positive'] = (
    (~best_rows['is_rebound_cluster'].astype(bool))
    & best_rows['predicted_state'].astype(bool)
    & (best_rows['target_lift'].abs() > config.min_effective_lift)
)
fp_lift_rows = best_rows[best_rows['effective_false_positive']].sort_values('target_lift', ascending=False)
fp_lift_rows.to_csv(OUT_DIR / 'effective_false_positive_lift_rows.csv', index=False)

print('guarded recipe scores')
display(recipe_scores.head(12))
print('effective false-positive lift rows')
display(fp_lift_rows)


guarded recipe scores


,feature_set,threshold,samples,positives,negatives,covered_folds,true_positive,false_positive,false_negative,true_negative,...,false_positive_rate,effective_false_positive,effective_false_positive_rate,false_positive_lift_mean,false_positive_lift_max,target_mae,rebound_target_mae,non_rebound_target_mae,recipe_gate,recipe_loss
0,momentum_drawdown,0.5,1261,7,1254,2,7,123,0,1131,...,0.098086,1,0.000797,0.001462,0.179855,0.000301,0.028571,0.000143,True,0.000000
1,momentum_drawdown,1.5,1261,7,1254,2,5,37,2,1217,...,0.029506,0,0.000000,0.000000,0.000000,0.000119,0.021429,0.000000,True,0.000000
2,momentum_soxx_relative,0.5,1261,7,1254,2,7,156,0,1098,...,0.124402,3,0.002392,0.003459,0.179855,0.000586,0.028571,0.000430,False,0.014110
3,momentum_soxx_relative,2.0,1261,7,1254,2,6,48,1,1206,...,0.038278,1,0.000797,0.003747,0.179855,0.000420,0.050000,0.000143,False,0.020000
4,momentum_drawdown,1.0,1261,7,1254,2,6,70,1,1184,...,0.055821,1,0.000797,0.002569,0.179855,0.000420,0.050000,0.000143,False,0.020000
5,momentum_soxx_relative,1.5,1261,7,1254,2,6,81,1,1173,...,0.064593,1,0.000797,0.002220,0.179855,0.000420,0.050000,0.000143,False,0.020000
6,momentum_soxx_relative,1.0,1261,7,1254,2,6,111,1,1143,...,0.088517,2,0.001595,0.003241,0.179855,0.000563,0.050000,0.000287,False,0.020000
7,momentum_drawdown,0.0,1261,7,1254,2,7,192,0,1062,...,0.153110,3,0.002392,0.002810,0.179855,0.000586,0.028571,0.000430,False,0.029648
8,momentum_drawdown,2.0,1261,7,1254,2,4,14,3,1240,...,0.011164,0,0.000000,0.000000,0.000000,0.000238,0.042857,0.000000,False,0.527143
9,momentum_soxx_relative,0.0,1261,7,1254,2,7,207,0,1047,...,0.165072,4,0.003190,0.006230,0.750000,0.001181,0.028571,0.001028,False,0.584579


effective false-positive lift rows


,row_index,feature_set,eval_fold,state_score,coverage_gate,train_positive,eval_positive,eval_negative,is_rebound_cluster,fold,true_target_position,default_target_position,rebound_target_position,threshold,predicted_state,proposed_target,target_lift,target_error,effective_false_positive
13220,610,momentum_drawdown,wf_2023_2024_train,1.21397,True,4,3,753,False,wf_2023_2024_train,0.25,0.25,0.429855,0.5,True,0.429855,0.179855,0.179855,True


In [3]:

best = recipe_scores.iloc[0].to_dict()
recipe_gate = bool(best.get('recipe_gate', False))
precision_caution = float(best.get('state_precision', np.nan)) < 0.10
lift_guard_pass = (
    float(best.get('effective_false_positive_rate', np.nan)) <= config.max_effective_false_positive_rate
    and float(best.get('false_positive_lift_mean', np.nan)) <= config.max_false_positive_lift_mean
    and float(best.get('false_positive_lift_max', np.nan)) <= config.max_false_positive_lift_max
)

if recipe_gate and lift_guard_pass:
    decision = 'state_head_lift_guard_passes_pre_ppo_no_promotion'
    next_mechanism = 'fold_local_ppo_aux_rebound_state_head_with_drawdown_guard'
    blocker = 'Lift guard passes only for pre-PPO recipe handoff; return/drawdown walk-forward gates remain untested.'
elif lift_guard_pass:
    decision = 'state_head_lift_guard_ok_but_recipe_gate_blocked'
    next_mechanism = 'calibrate_state_head_threshold_before_ppo'
    blocker = 'Effective false-positive lift is bounded, but recall/precision/MAE gate did not pass.'
else:
    decision = 'state_head_lift_guard_failed_no_ppo_handoff'
    next_mechanism = 'tighten_rebound_state_head_or_add_risk_filter'
    blocker = 'False-positive lift is too large for a safe PPO recipe.'

selection = {
    'experiment': EXPERIMENT,
    'created_at': base_context['created_at'],
    'source_experiment': 'rebound_state_head_target_recipe_v1',
    'direct_comparison_baseline': 'rebound_representation_separability_v1',
    'decision': decision,
    'promotion_allowed': False,
    'full_gate': False,
    'recipe_gate': recipe_gate,
    'lift_guard_pass': bool(lift_guard_pass),
    'precision_caution': bool(precision_caution),
    'promotion_blocker': blocker,
    'best_feature_set': str(best.get('feature_set', '')),
    'best_threshold': float(best.get('threshold', np.nan)),
    'best_state_recall': float(best.get('state_recall', np.nan)),
    'best_state_precision': float(best.get('state_precision', np.nan)),
    'best_false_positive_rate': float(best.get('false_positive_rate', np.nan)),
    'best_effective_false_positive': int(best.get('effective_false_positive', 0)),
    'best_effective_false_positive_rate': float(best.get('effective_false_positive_rate', np.nan)),
    'best_false_positive_lift_mean': float(best.get('false_positive_lift_mean', np.nan)),
    'best_false_positive_lift_max': float(best.get('false_positive_lift_max', np.nan)),
    'best_target_mae': float(best.get('target_mae', np.nan)),
    'best_rebound_target_mae': float(best.get('rebound_target_mae', np.nan)),
    'next_mechanism': next_mechanism,
    'acceptance_gate_for_next_run': {
        'relative_return_min_pp': 30.0,
        'max_dd_max_pct': 15.0,
        'fixed_split_required': True,
        'walkforward_required': True,
        'state_head_lift_guard_required': True,
        'precision_caution_requires_ppo_drawdown_guard': True,
        'no_runtime_promotion_from_recipe_only': True,
    },
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://arxiv.org/abs/2004.11362',
        'https://arxiv.org/abs/1807.03748',
        'https://arxiv.org/abs/1901.05555',
    ],
    'sidecars': [
        'base_context.json',
        'guarded_recipe_scores.csv',
        'effective_false_positive_lift_rows.csv',
        'summary.json',
    ],
}
summary = {
    'base_context': base_context,
    'best_row': best,
    'selection': selection,
    'conclusion': [
        f"Best guarded recipe {selection['best_feature_set']} threshold={selection['best_threshold']} has recall/precision/FPR {selection['best_state_recall']:.3f}/{selection['best_state_precision']:.3f}/{selection['best_false_positive_rate']:.3f}.",
        f"Effective false positives={selection['best_effective_false_positive']} with rate {selection['best_effective_false_positive_rate']:.4f}; lift mean/max {selection['best_false_positive_lift_mean']:.4f}/{selection['best_false_positive_lift_max']:.4f}.",
        f"Decision: {selection['decision']}; next mechanism: {selection['next_mechanism']}.",
        "This remains a pre-PPO handoff only; low precision still requires drawdown/risk guard and full walk-forward validation.",
    ],
}

(OUT_DIR / 'base_context.json').write_text(json.dumps(base_context, indent=2, sort_keys=True, default=str) + '\n')
(OUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2, sort_keys=True, default=str) + '\n')
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True, default=str) + '\n')
print(json.dumps(selection, indent=2, sort_keys=True))


{
  "acceptance_gate_for_next_run": {
    "fixed_split_required": true,
    "max_dd_max_pct": 15.0,
    "no_runtime_promotion_from_recipe_only": true,
    "precision_caution_requires_ppo_drawdown_guard": true,
    "relative_return_min_pp": 30.0,
    "state_head_lift_guard_required": true,
    "walkforward_required": true
  },
  "best_effective_false_positive": 1,
  "best_effective_false_positive_rate": 0.0007974481658692185,
  "best_false_positive_lift_max": 0.17985462658059442,
  "best_false_positive_lift_mean": 0.001462232736427597,
  "best_false_positive_rate": 0.09808612440191387,
  "best_feature_set": "momentum_drawdown",
  "best_rebound_target_mae": 0.028571428571428574,
  "best_state_precision": 0.05384615384615385,
  "best_state_recall": 1.0,
  "best_target_mae": 0.00030123285216541985,
  "best_threshold": 0.5,
  "created_at": "2026-05-25T13:53:31.630824+00:00",
  "decision": "state_head_lift_guard_passes_pre_ppo_no_promotion",
  "direct_comparison_baseline": "rebound_represent

## Conclusion

This notebook is the single-run conclusion for effective false-positive lift risk. Passing this screen can only authorize a future fold-local PPO recipe with auxiliary rebound state head and drawdown guard; it cannot promote runtime behavior.
